# GraphRAG Embedding & Similarity Calculation Demo

This notebook demonstrates how GraphRAG calculates embeddings and similarity scores between a query and documents.

## Overview
- **Embedding Model**: Same as GraphRAG's configuration (`text-embedding-3-large` on Azure OpenAI)
- **Similarity Metric**: Cosine similarity (used by GraphRAG's vector stores)
- **Use Case**: Finding which document is most relevant to a query
- **API**: Azure OpenAI (matching your GraphRAG setup)

In [ ]:
# Import required libraries
import os
import numpy as np
from openai import AzureOpenAI
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv("/home/chuaxu/projects/graphrag/ragtest/.env")

# Initialize Azure OpenAI client (same as GraphRAG)
client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version="2024-10-21",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT")
)

print("✅ Azure OpenAI client initialized")
print(f"🔑 API Key loaded: {os.getenv('AZURE_OPENAI_API_KEY')[:20]}...")
print(f"🌐 Endpoint: {os.getenv('AZURE_OPENAI_ENDPOINT')}")

## Define Documents and Query

These are our test documents and the query we want to match against them.

In [ ]:
# Documents
# documents = [
#     "The EU announced the Green Deal in 2019.",
#     "The Green Deal aims to make Europe carbon-neutral.",
#     "Carbon neutrality targets reduce fossil fuel usage."
# ]
documents = [
    "New York's College Savings Program offers tax-deferred growth but does not pay fixed interest like a saving account. Returns depend on investment options.",
    "The average interest rate on savings accounts in New York is currently 0.4%.",
    "A 529 plan is a tax-advantaged savings plan designed to encourage saving for future education costs."
]

# Query
# query = "When did the EU announce its climate plan?"
query = "What’s the average interest rate for a 529 plan in New York?"

print("📄 Documents:")
for i, doc in enumerate(documents, 1):
    print(f"   {i}. {doc}")

print(f"\n❓ Query: {query}")

## Step 1: Generate Embeddings

GraphRAG uses Azure OpenAI's embedding model. In your configuration, it's set to `text-embedding-3-large`. This is the same embedding model used in:
- Entity description embeddings
- Text unit embeddings
- Query embeddings for local search

**Note**: The deployment name must match what's configured in your Azure OpenAI resource.

In [ ]:
# GraphRAG's configured embedding model (from your settings.yaml)
EMBEDDING_MODEL = "text-embedding-3-large"  # The model name
DEPLOYMENT_NAME = "text-embedding-3-large"  # Azure deployment name

def get_embedding(text: str) -> list[float]:
    """
    Get embedding for a text using Azure OpenAI's API.
    This is the same method GraphRAG uses internally.
    """
    response = client.embeddings.create(
        model=DEPLOYMENT_NAME,  # Use deployment name for Azure
        input=text
    )
    return response.data[0].embedding

# Generate embeddings for all documents
print("🔄 Generating embeddings for documents...")
doc_embeddings = [get_embedding(doc) for doc in documents]

# Generate embedding for query
print("🔄 Generating embedding for query...")
query_embedding = get_embedding(query)

print(f"\n✅ Embeddings generated!")
print(f"   - Embedding model: {EMBEDDING_MODEL}")
print(f"   - Embedding dimension: {len(query_embedding)}")
print(f"   - Document embeddings: {len(doc_embeddings)}")
print(f"   - Query embedding dimension: {len(query_embedding)}")

### Inspect the Embedding Vectors

Let's look at the actual embedding vectors generated for each text. These are high-dimensional numerical representations (3072 dimensions for text-embedding-3-large).

In [ ]:
print("="*80)
print("📊 EMBEDDING VECTORS OVERVIEW")
print("="*80)

# Display query embedding
print(f"\n❓ QUERY: \"{query}\"")
print("-"*80)
print(f"Embedding dimension: {len(query_embedding)}")
print(f"First 10 values: {[f'{v:.6f}' for v in query_embedding[:10]]}")
print(f"Last 10 values: {[f'{v:.6f}' for v in query_embedding[-10:]]}")
print(f"Min value: {min(query_embedding):.6f}")
print(f"Max value: {max(query_embedding):.6f}")
print(f"Mean value: {np.mean(query_embedding):.6f}")
print(f"Std deviation: {np.std(query_embedding):.6f}")

# Display document embeddings
for i, (doc, emb) in enumerate(zip(documents, doc_embeddings), 1):
    print(f"\n📄 DOCUMENT {i}: \"{doc}\"")
    print("-"*80)
    print(f"Embedding dimension: {len(emb)}")
    print(f"First 10 values: {[f'{v:.6f}' for v in emb[:10]]}")
    print(f"Last 10 values: {[f'{v:.6f}' for v in emb[-10:]]}")
    print(f"Min value: {min(emb):.6f}")
    print(f"Max value: {max(emb):.6f}")
    print(f"Mean value: {np.mean(emb):.6f}")
    print(f"Std deviation: {np.std(emb):.6f}")

print("\n" + "="*80)
print("💡 INSIGHTS")
print("="*80)
print("""
- Each embedding is a {}-dimensional vector of floating-point numbers
- Values are typically normalized (range roughly -1 to 1)
- Similar texts have vectors pointing in similar directions (high cosine similarity)
- The embedding captures semantic meaning, not just word overlap
- These high-dimensional vectors are what get compared during search
""".format(len(query_embedding)))

print("="*80)

### Full Embedding Vectors (Optional)

If you want to see the complete embedding vectors, run the cell below. Warning: This will generate a lot of output (3072 numbers per text)!

In [ ]:
# Set to True to see full vectors, False to see truncated view
SHOW_FULL_VECTORS = False  # Change to True if you want to see all 3072 values

if SHOW_FULL_VECTORS:
    print("="*80)
    print("🔢 COMPLETE EMBEDDING VECTORS")
    print("="*80)
    
    print(f"\n❓ Query Embedding ({len(query_embedding)} dimensions):")
    print(query_embedding)
    
    for i, (doc, emb) in enumerate(zip(documents, doc_embeddings), 1):
        print(f"\n📄 Document {i} Embedding ({len(emb)} dimensions):")
        print(f"Text: \"{doc}\"")
        print(emb)
    
    print("\n" + "="*80)
else:
    # Show truncated view with more context
    print("="*80)
    print("🔢 EMBEDDING VECTORS (Truncated View)")
    print("="*80)
    print(f"Set SHOW_FULL_VECTORS = True to see all {len(query_embedding)} dimensions")
    
    print(f"\n❓ Query: \"{query}\"")
    print(f"Embedding (first 50 of {len(query_embedding)} dimensions):")
    print(f"  {query_embedding[:50]}")
    print(f"  ...")
    
    for i, (doc, emb) in enumerate(zip(documents, doc_embeddings), 1):
        print(f"\n📄 Document {i}: \"{doc}\"")
        print(f"Embedding (first 50 of {len(emb)} dimensions):")
        print(f"  {emb[:50]}")
        print(f"  ...")
    
    print("\n" + "="*80)
    print("💡 TIP: These vectors are compared using cosine similarity")
    print("   to find semantically similar content in GraphRAG.")
    print("="*80)

## Step 2: Calculate Cosine Similarity

GraphRAG uses **cosine similarity** to measure how similar two embeddings are:

$$
\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}
$$

- **Range**: -1 to 1 (but typically 0 to 1 for text embeddings)
- **1.0**: Identical/very similar
- **0.0**: Unrelated
- **Higher score**: More relevant to the query

In [ ]:
def cosine_similarity(vec1: list[float], vec2: list[float]) -> float:
    """
    Calculate cosine similarity between two vectors.
    This is the same metric used by GraphRAG's vector stores (LanceDB, etc.)
    """
    vec1 = np.array(vec1)
    vec2 = np.array(vec2)
    
    # Cosine similarity formula
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    
    similarity = dot_product / (norm_vec1 * norm_vec2)
    return float(similarity)

# Calculate similarity between query and each document
print("📊 Calculating cosine similarity scores...\n")
similarities = []

for i, (doc, doc_emb) in enumerate(zip(documents, doc_embeddings), 1):
    similarity = cosine_similarity(query_embedding, doc_emb)
    similarities.append((i, doc, similarity))
    print(f"Document {i}: {doc}")
    print(f"   Similarity Score: {similarity:.6f}")
    print()

## Step 3: Rank Documents by Relevance

This is how GraphRAG ranks entities or text units during local search:
1. Embed the query
2. Calculate similarity with all candidates
3. Sort by similarity (highest first)
4. Select top-k results

In [ ]:
# Sort by similarity (highest first)
ranked_docs = sorted(similarities, key=lambda x: x[2], reverse=True)

print("="*80)
print("🏆 RANKED RESULTS (Most Relevant First)")
print("="*80)

for rank, (doc_id, doc, score) in enumerate(ranked_docs, 1):
    # Visual representation of similarity
    bar_length = int(score * 50)  # Scale to 50 chars max
    bar = "█" * bar_length
    
    print(f"\nRank #{rank}")
    print(f"Document {doc_id}: \"{doc}\"")
    print(f"Similarity: {score:.6f}  {bar}")
    
    # Interpret the score
    if score > 0.8:
        relevance = "🔥 Highly Relevant"
    elif score > 0.6:
        relevance = "✅ Relevant"
    elif score > 0.4:
        relevance = "⚠️  Somewhat Relevant"
    else:
        relevance = "❌ Not Very Relevant"
    print(f"Relevance: {relevance}")

print("\n" + "="*80)

## Analysis: Why This Ranking?

Let's analyze why the query matched each document the way it did.

In [ ]:
print("="*80)
print("🔍 SEMANTIC SIMILARITY ANALYSIS")
print("="*80)

print(f"\n❓ Query: \"{query}\"")
print("\nKey query terms: EU, announce, climate plan, when (temporal)")

print("\n" + "-"*80)
print("📄 Document 1: \"The EU announced the Green Deal in 2019.\"")
print("-"*80)
print("Matching concepts:")
print("   ✅ 'EU' - EXACT match")
print("   ✅ 'announced' - EXACT match (same verb as 'announce')")
print("   ✅ 'Green Deal' - Related to 'climate plan' (semantic match)")
print("   ✅ '2019' - Answers 'when' (temporal information)")
print("\n🎯 Expected: HIGHEST similarity (most relevant)")

print("\n" + "-"*80)
print("📄 Document 2: \"The Green Deal aims to make Europe carbon-neutral.\"")
print("-"*80)
print("Matching concepts:")
print("   ✅ 'Green Deal' - Related to 'climate plan'")
print("   ✅ 'Europe' - Related to 'EU' (geographic entity)")
print("   ⚠️  No 'announce' or temporal information")
print("   ⚠️  Describes goals, not announcement")
print("\n🎯 Expected: MEDIUM similarity (topically related but doesn't answer 'when')")

print("\n" + "-"*80)
print("📄 Document 3: \"Carbon neutrality targets reduce fossil fuel usage.\"")
print("-"*80)
print("Matching concepts:")
print("   ⚠️  'Carbon neutrality' - Loosely related to climate topics")
print("   ❌ No 'EU', 'announce', or 'Green Deal'")
print("   ❌ No temporal information")
print("   ❌ Very general statement about climate goals")
print("\n🎯 Expected: LOWEST similarity (tangentially related)")

print("\n" + "="*80)

## How GraphRAG Uses This in Practice

### In Local Search:
1. **Query comes in**: "When did the EU announce its climate plan?"
2. **Embed the query** using `text-embedding-3-small`
3. **Search vector store** for similar entity descriptions:
   - Calculate cosine similarity with all entity embeddings
   - Return top-k entities (default: k=10)
   - Example: `EU`, `GREEN DEAL`, `2019` entities would rank high
4. **Build context** from selected entities and their relationships
5. **Send to LLM** with the context

### Key Parameters in GraphRAG:
```python
# From LocalSearch configuration
top_k_mapped_entities = 10        # Select top 10 most similar entities
oversample_scaler = 2             # Retrieve 20, filter to 10
embedding_vectorstore_key = ID    # Use entity ID as vector store key
```

### Similarity Threshold:
- GraphRAG doesn't use a hard threshold by default
- Instead, it uses **top-k selection** (relative ranking)
- Higher similarity = higher priority in context
- Token budget allocation prioritizes high-similarity entities

## Bonus: Visualize Embedding Space

Let's use dimensionality reduction (t-SNE) to visualize how the embeddings relate in 2D space.

In [ ]:
# Optional: Install required libraries if not already available
try:
    from sklearn.manifold import TSNE
    import matplotlib.pyplot as plt
    HAS_VISUALIZATION = True
except ImportError:
    HAS_VISUALIZATION = False
    print("⚠️  scikit-learn or matplotlib not installed. Skipping visualization.")
    print("   Install with: pip install scikit-learn matplotlib")

if HAS_VISUALIZATION:
    # Combine all embeddings and convert to numpy array
    all_embeddings = np.array(doc_embeddings + [query_embedding])
    labels = [f"Doc {i+1}" for i in range(len(documents))] + ["Query"]
    
    print(f"📊 Preparing visualization...")
    print(f"   - Total embeddings: {len(all_embeddings)}")
    print(f"   - Embedding dimension: {all_embeddings.shape[1]}")
    
    # Reduce to 2D using t-SNE
    # Note: perplexity must be less than n_samples
    perplexity = min(2, len(all_embeddings) - 1)
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    embeddings_2d = tsne.fit_transform(all_embeddings)
    
    print(f"   - Reduced to 2D using t-SNE (perplexity={perplexity})")
    
    # Plot
    plt.figure(figsize=(10, 8))
    
    # Plot documents
    for i in range(len(documents)):
        plt.scatter(embeddings_2d[i, 0], embeddings_2d[i, 1], 
                   s=200, alpha=0.6, label=labels[i])
        plt.annotate(labels[i], (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                    fontsize=10, ha='center')
    
    # Plot query with different style
    plt.scatter(embeddings_2d[-1, 0], embeddings_2d[-1, 1], 
               s=300, marker='*', c='red', label='Query', edgecolors='black', linewidth=2)
    plt.annotate('Query', (embeddings_2d[-1, 0], embeddings_2d[-1, 1]),
                fontsize=12, ha='center', fontweight='bold')
    
    plt.title('Embedding Space Visualization (t-SNE)', fontsize=14, fontweight='bold')
    plt.xlabel('Dimension 1')
    plt.ylabel('Dimension 2')
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\n📊 Visualization shows:")
    print("   - Closer points = More similar semantically")
    print("   - Query (red star) should be closest to most relevant document")
    print("   - Document 1 should be nearest to Query (highest similarity)")

## Summary

### Key Takeaways:

1. **Embedding Model**: `text-embedding-3-large` (Your GraphRAG configuration)
   - Dimension: 3072 (for text-embedding-3-large)
   - Captures semantic meaning of text
   - Hosted on Azure OpenAI

2. **Similarity Metric**: Cosine Similarity
   - Range: 0 to 1 (for text embeddings)
   - Measures angle between vectors, not magnitude
   - Used by GraphRAG's vector stores (LanceDB, etc.)

3. **Ranking Strategy**: Top-K Selection
   - Sort by similarity (descending)
   - Select top-k most similar items
   - GraphRAG uses this for entity selection in local search

4. **Why Document 1 Wins**:
   - Exact matches: "EU", "announced"
   - Semantic matches: "Green Deal" ≈ "climate plan"
   - Temporal information: "2019" answers "when"

### GraphRAG Usage:
- **Entity Selection**: Embed query → Find similar entity descriptions
- **Text Unit Ranking**: Embed query → Find similar text units
- **Context Building**: Prioritize high-similarity entities for LLM context
- **Token Budget**: Allocate more tokens to highly relevant entities

### Configuration:
```yaml
# In settings.yaml (your actual configuration)
models:
  default_embedding_model:
    type: azure_openai_embedding
    api_base: https://ard-genai-copilot-openai.openai.azure.com/
    model: text-embedding-3-large
    deployment_name: text-embedding-3-large
  
local_search:
  top_k_mapped_entities: 10         # Top 10 similar entities
  oversample_scaler: 2              # Retrieve 2x, then filter
```